In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10

(x_train, _), (_, _) = cifar10.load_data()

# Normalize to [-1, 1]
x_train = (x_train.astype("float32") - 127.5) / 127.5

BUFFER_SIZE = 60000
BATCH_SIZE = 128

dataset = tf.data.Dataset.from_tensor_slices(x_train)\
    .shuffle(BUFFER_SIZE)\
    .batch(BATCH_SIZE)

latent_dim = 100

# Generator
def build_generator():
    model = tf.keras.Sequential()

    model.add(layers.Dense(8*8*256, use_bias=False, input_shape=(latent_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((8, 8, 256)))

    model.add(layers.Conv2DTranspose(128, 4, strides=2, padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, 4, strides=2, padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(3, 3, strides=1, padding='same', activation='tanh'))

    return model


# Discriminator
def build_discriminator():
    model = tf.keras.Sequential()

    model.add(layers.Conv2D(64, 4, strides=2, padding='same', input_shape=[32,32,3]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, 4, strides=2, padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model


generator = build_generator()
discriminator = build_discriminator()

cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss


generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)


@tf.function
def train_step(images):
    batch_size = tf.shape(images)[0]
    noise = tf.random.normal([batch_size, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))


EPOCHS = 20

for epoch in range(EPOCHS):
    for image_batch in dataset:
        train_step(image_batch)

    print(f"Epoch {epoch+1} completed")


# Generate images
noise = tf.random.normal([10, latent_dim])
generated_images = generator(noise, training=False)
generated_images = (generated_images + 1) / 2

plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(generated_images[i])
    plt.axis("off")

plt.suptitle("Generated Artistic Images")
plt.show()


# Latent interpolation
z1 = tf.random.normal([1, latent_dim])
z2 = tf.random.normal([1, latent_dim])

alphas = np.linspace(0,1,10)
images = []

for a in alphas:
    z = (1-a)*z1 + a*z2
    img = generator(z, training=False)
    images.append(img[0])

images = (np.array(images)+1)/2

plt.figure(figsize=(12,3))
for i, img in enumerate(images):
    plt.subplot(1,10,i+1)
    plt.imshow(img)
    plt.axis("off")

plt.suptitle("Latent Space Interpolation")
plt.show()

In [ ]:
# Run training + generate images (new cell at index 1)
# Requires the definitions from the previous cell (dataset, train_step, EPOCHS, generator, latent_dim, files)
if 'train_step' not in globals() or 'dataset' not in globals() or 'generator' not in globals():
    raise RuntimeError("Required objects not found. Run the previous cell (index 0) first.")

for epoch in range(EPOCHS):
    for image_batch in dataset:
        train_step(image_batch)
    print(f"Epoch {epoch+1} completed")
print("Training Finished")

# Generate and display images
noise = np.random.normal(0, 1, (10, latent_dim))
generated_images = generator.predict(noise)

plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    img = (generated_images[i] + 1) / 2.0
    plt.imshow(img)
    plt.axis("off")
plt.show()

# Save and attempt to download the first image
img = (generated_images[0] + 1) / 2.0
plt.imsave("gan_artistic_output.png", img)
try:
    files.download("gan_artistic_output.png")
except Exception:
    print("files.download not available in this environment; image saved to gan_artistic_output.png")